In [1]:
import torch

print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


print("torch:", torch.__version__)
print("cuda version:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())


1
NVIDIA GeForce RTX 4080 SUPER
torch: 2.11.0+cu128
cuda version: 12.8
cuda available: True


In [2]:
import os
import uuid
import queue
import threading
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import psutil
import pypdf
import spacy
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct
import warnings
from pypdf.errors import PdfReadWarning
import logging



from qdrant_client.models import VectorParams, Distance
# Requires transformers>=4.51.0
# Requires sentence-transformers>=2.7.0

from sentence_transformers import SentenceTransformer
import threading, time


# # Load the model
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device='cuda')

# 2. Initialize Models and Client
print("Initializing environment components...")
nlp_model = spacy.load("en_core_web_sm")
embedder_model = model
qdrant_client = QdrantClient(url="http://localhost:6333")

# 3. Setup Qdrant Collection Structure
print("Re-creating Qdrant collection 'ai_knowledge'...")
qdrant_client.recreate_collection(
    collection_name="ai_knowledge",
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE)
)

file_paths = [os.path.join(os.getcwd(),"ai_pdfs",f) for f in os.listdir(os.path.join(os.getcwd(), "ai_pdfs"))]

c:\Users\Owner\anaconda3\envs\mlstuff\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 9109.24it/s]


Initializing environment components...
Re-creating Qdrant collection 'ai_knowledge'...


C:\Users\Owner\AppData\Local\Temp\ipykernel_25032\747065412.py:40: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


In [3]:

text = "Hello world. This is a test sentence. Here is another."
t0 = time.perf_counter()
list(nlp_model(text).sents)
print(f"{(time.perf_counter()-t0)*1000:.1f}ms")  # should be <5ms

8.3ms


In [4]:
import multiprocessing
from multiprocessing import Process, Queue as MPQueue
import os
import queue
import threading
import time
import uuid
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import psutil
import pypdf
import torch
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct

# ── Tunable constants ──────────────────────────────────────────────────────────
EMBED_BATCH_SIZE  = 128
UPSERT_BATCH_SIZE = 32
NLP_WORKERS       = 4
CHUNK_WORKERS     = 2
COSINE_THRESHOLD  = 0.7
COLLECTION_NAME   = "ai_knowledge"
MONITOR_INTERVAL  = 2.0
MAX_TOKENS        = 512

_t0 = time.perf_counter()

def trace(stage: str, msg: str) -> None:
    elapsed = time.perf_counter() - _t0
    thread  = threading.current_thread().name
    print(f"[{elapsed:>8.3f}s] [{thread:<20}] [{stage:<12}] {msg}", flush=True)


# ─────────────────────────────────────────────────────────────────────────────
# Stage 0 — Monitor
# ─────────────────────────────────────────────────────────────────────────────
def _monitor_worker(sent_q, embed_out_q, upsert_q, stop_event, interval=MONITOR_INTERVAL):
    proc     = psutil.Process(os.getpid())
    has_cuda = torch.cuda.is_available()

    while not stop_event.is_set():
        # MPQueue.qsize() raises NotImplementedError on macOS — guard it
        try:
            sq = sent_q.qsize()
        except NotImplementedError:
            sq = -1
        try:
            eq = embed_out_q.qsize()
        except NotImplementedError:
            eq = -1
        uq = upsert_q.qsize()

        if has_cuda:
            alloc    = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved()  / 1024**3
            total    = torch.cuda.get_device_properties(0).total_memory / 1024**3
            vram_str = f"VRAM {alloc:.2f}/{reserved:.2f}/{total:.1f} GB (alloc/reserv/total)"
        else:
            vram_str = "VRAM N/A (no CUDA)"

        ram_gb = proc.memory_info().rss / 1024**3
        print(
            f"[MON] sent_q={sq}  embed_q={eq}  upsert_q={uq} | {vram_str} | RAM {ram_gb:.2f} GB",
            flush=True,
        )
        stop_event.wait(interval)


# ─────────────────────────────────────────────────────────────────────────────
# Stage 1 — NLP Workers
# ─────────────────────────────────────────────────────────────────────────────
_thread_local = threading.local()

def _nlp_initializer(model_name: str) -> None:
    """Called once per worker thread — each thread gets its own spaCy instance."""
    import spacy
    _thread_local.nlp = spacy.load(model_name)

def _nlp_worker(sent_q, error_event, file_path, page_num, raw) -> None:
    fname = os.path.basename(file_path)
    trace("NLP", f"START {fname} p{page_num} ({len(raw)} chars)")

    if error_event.is_set():
        trace("NLP", f"SKIP {fname} p{page_num} — error_event set")
        return
    if not raw.strip():
        trace("NLP", f"SKIP {fname} p{page_num} — empty")
        return

    nlp = _thread_local.nlp
    t0  = time.perf_counter()

    # Parse once, reuse the doc — was previously called twice (bug)
    doc = nlp(raw)
    trace("NLP", f"spaCy took {(time.perf_counter()-t0)*1000:.1f}ms")

    pos = 0
    for sent in doc.sents:          # ← doc, not nlp(raw) again
        text = sent.text.strip()
        if not text:
            continue
        tokens = text.split()
        chunks = (
            [" ".join(tokens[i:i + MAX_TOKENS]) for i in range(0, len(tokens), MAX_TOKENS)]
            if len(tokens) > MAX_TOKENS else [text]
        )
        for chunk in chunks:
            trace("NLP", f"  PUT sent_q pos={pos}: '{chunk[:40]}...'")
            sent_q.put({"doc_id": fname, "page": page_num + 1, "pos": pos, "text": chunk})
            pos += 1

    trace("NLP", f"DONE {fname} p{page_num} — {pos} sentences in {time.perf_counter()-t0:.3f}s")


# ─────────────────────────────────────────────────────────────────────────────
# Stage 2 — Embedder  (runs in a child process — separate GIL)
# ─────────────────────────────────────────────────────────────────────────────
def _embedder_worker(embedder, sent_q, embed_out_q, n_chunk_workers, error_event) -> None:
    trace("EMBEDDER", "Started — waiting for sentences")
    pending = []

    def _flush(items):
        trace("EMBEDDER", f"Flushing {len(items)} sentences to GPU")
        texts = [it["text"] for it in items]
        with torch.inference_mode():
            vecs = embedder.encode(
                texts,
                batch_size=EMBED_BATCH_SIZE,
                show_progress_bar=False,
                convert_to_numpy=True,
            )
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        groups = defaultdict(list)
        for item, vec in zip(items, vecs):
            groups[item["doc_id"]].append({**item, "vector": vec})
        for doc_items in groups.values():
            doc_items.sort(key=lambda x: x["pos"])
            trace("EMBEDDER", f"  PUT embed_out_q: {len(doc_items)} items for {doc_items[0]['doc_id']}")
            embed_out_q.put(doc_items)
        trace("EMBEDDER", "Flush complete")

    try:
        while True:
            try:
                item = sent_q.get(timeout=2.0)
            except queue.Empty:
                trace("EMBEDDER", f"Queue idle — pending={len(pending)}")
                if pending:
                    _flush(pending)
                    pending = []
                continue

            if item is None:
                trace("EMBEDDER", f"Got sentinel — flushing {len(pending)} remaining")
                if pending:
                    _flush(pending)
                break

            pending.append(item)
            trace("EMBEDDER", f"Got sentence #{len(pending)}")
            if len(pending) >= EMBED_BATCH_SIZE:
                _flush(pending)
                pending = []

    except Exception as e:
        trace("EMBEDDER", f"CRASHED: {type(e).__name__}: {e}")
        error_event.set()
    finally:
        for i in range(n_chunk_workers):
            trace("EMBEDDER", f"Sending sentinel {i+1}/{n_chunk_workers} to embed_out_q")
            embed_out_q.put(None)
        trace("EMBEDDER", "Done")


def _embedder_process_entry(embed_model_name, sent_q, embed_out_q, chunk_workers, error_event) -> None:
    """Entry point for the embedder child process."""
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer(embed_model_name)
    _embedder_worker(embedder, sent_q, embed_out_q, chunk_workers, error_event)


# ─────────────────────────────────────────────────────────────────────────────
# Stage 3 — Chunking Workers
# ─────────────────────────────────────────────────────────────────────────────
def _chunking_worker(embed_out_q, upsert_q) -> None:
    while True:
        doc_items = embed_out_q.get()
        if doc_items is None:
            break
        _split_and_enqueue(doc_items, upsert_q)


def _split_and_enqueue(doc_items: list[dict], upsert_q) -> None:
    vecs = np.array([it["vector"] for it in doc_items], dtype=np.float32)

    if len(doc_items) == 1:
        _push_chunk(doc_items, vecs, 0, 1, upsert_q)
        return

    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    norm_vecs = vecs / norms

    cosines   = np.sum(norm_vecs[:-1] * norm_vecs[1:], axis=1)
    distances = 1.0 - cosines
    splits    = np.where(distances > COSINE_THRESHOLD)[0]

    boundaries = [0, *(int(s) + 1 for s in splits), len(doc_items)]
    for i in range(len(boundaries) - 1):
        _push_chunk(doc_items, vecs, boundaries[i], boundaries[i + 1], upsert_q)


def _push_chunk(doc_items, vecs, start, end, upsert_q) -> None:
    segment    = doc_items[start:end]
    chunk_text = " ".join(it["text"] for it in segment)
    chunk_vec  = np.mean(vecs[start:end], axis=0)
    norm       = np.linalg.norm(chunk_vec)
    if norm > 0:
        chunk_vec /= norm

    upsert_q.put(PointStruct(
        id=str(uuid.uuid4()),
        vector=chunk_vec.tolist(),
        payload={
            "text":        chunk_text,
            "source_file": segment[0]["doc_id"],
            "page_n":      segment[0]["page"],
        },
    ))


# ─────────────────────────────────────────────────────────────────────────────
# Stage 4 — Qdrant Upsert Worker
# ─────────────────────────────────────────────────────────────────────────────
def _upsert_worker(vector_db: QdrantClient, upsert_q: queue.Queue) -> None:
    batch: list[PointStruct] = []
    total = 0

    def _flush() -> None:
        nonlocal total
        vector_db.upsert(collection_name=COLLECTION_NAME, wait=True, points=batch)
        total += len(batch)
        batch.clear()

    while True:
        item = upsert_q.get()
        if item is None:
            if batch:
                _flush()
            break
        batch.append(item)
        if len(batch) >= UPSERT_BATCH_SIZE:
            _flush()

    print(f"[Qdrant] Done. Total chunks: {total}")


# ─────────────────────────────────────────────────────────────────────────────
# PDF Iterator
# ─────────────────────────────────────────────────────────────────────────────
def get_pdf_tasks(pdf_paths):
    for file_path in pdf_paths:
        try:
            with open(file_path, "rb") as fh:
                reader = pypdf.PdfReader(fh, strict=False)
                for page_num, page in enumerate(reader.pages):
                    text = page.extract_text() or ""
                    yield file_path, page_num, text
        except Exception as e:
            trace("ITERATOR", f"FAILED on {file_path}: {e}")


# ─────────────────────────────────────────────────────────────────────────────
# Main Pipeline
# ─────────────────────────────────────────────────────────────────────────────
def ingest_pdfs(
    pdf_paths: list[str],
    vector_db: QdrantClient,
    embed_model_name: str,          # e.g. "sentence-transformers/all-MiniLM-L6-v2"
    nlp_model_name: str = "en_core_web_sm",
) -> None:
    """
    Thread/process layout:
      1 monitor thread      — queue depth + memory dashboard
      NLP_WORKERS threads   — pypdf + spaCy (CPU; own nlp instance per thread)
      1 embedder PROCESS    — GPU encode (separate GIL; never freezes NLP threads)
      CHUNK_WORKERS threads — cosine sim (CPU; numpy releases GIL)
      1 upsert thread       — Qdrant network I/O

    Queue boundaries:
      sent_q      MPQueue  — crosses process boundary (NLP threads → embedder process)
      embed_out_q MPQueue  — crosses process boundary (embedder process → chunker threads)
      upsert_q    Queue    — stays in main process   (chunker threads → upsert thread)
    """
    # Queues that cross the process boundary must be MPQueue
    sent_q      = MPQueue(maxsize=768)
    embed_out_q = MPQueue(maxsize=20)
    # Stays in-process — regular Queue is faster
    upsert_q    = queue.Queue(maxsize=500)
    # Must be multiprocessing.Event so the child process can read/set it
    error_event = multiprocessing.Event()

    # ── Monitor ───────────────────────────────────────────────────────────────
    stop_monitor   = threading.Event()
    monitor_thread = threading.Thread(
        target=_monitor_worker,
        args=(sent_q, embed_out_q, upsert_q, stop_monitor),
        name="monitor",
        daemon=True,
    )
    monitor_thread.start()

    # ── Downstream workers (start before NLP so they're ready to consume) ────
    chunk_threads = [
        threading.Thread(
            target=_chunking_worker,
            args=(embed_out_q, upsert_q),
            name=f"chunker-{i}",
            daemon=True,
        )
        for i in range(CHUNK_WORKERS)
    ]
    upsert_thread = threading.Thread(
        target=_upsert_worker,
        args=(vector_db, upsert_q),
        name="qdrant",
        daemon=True,
    )
    embed_proc = Process(
        target=_embedder_process_entry,
        args=(embed_model_name, sent_q, embed_out_q, CHUNK_WORKERS, error_event),
        name="embedder",
        daemon=True,
    )

    for t in chunk_threads:
        t.start()
    upsert_thread.start()
    embed_proc.start()

    # ── NLP fan-out ───────────────────────────────────────────────────────────
    trace("ORCH", f"Submitting tasks to ThreadPoolExecutor (NLP_WORKERS={NLP_WORKERS})")

    with ThreadPoolExecutor(
        max_workers=NLP_WORKERS,
        thread_name_prefix="nlp",
        initializer=_nlp_initializer,
        initargs=(nlp_model_name,),
    ) as pool:
        futures = {}
        for fp, pn, text in get_pdf_tasks(pdf_paths):
            if error_event.is_set():
                break
            fut = pool.submit(_nlp_worker, sent_q, error_event, fp, pn, text)
            futures[fut] = (fp, pn)

        failed = 0
        for fut in futures:
            try:
                fut.result()
            except Exception as e:
                failed += 1
                trace("ORCH", f"Task FAILED {futures[fut]}: {repr(e)}")

    if failed:
        trace("ORCH", f"{failed}/{len(futures)} NLP tasks failed")

    # ── Sentinel propagation ──────────────────────────────────────────────────
    sent_q.put(None)          # → embedder drains + sends CHUNK_WORKERS Nones to embed_out_q
    embed_proc.join()         # wait for embedder to finish

    for t in chunk_threads:
        t.join()              # each chunker exits on its own None

    upsert_q.put(None)        # → upsert worker flushes remainder and exits
    upsert_thread.join()

    stop_monitor.set()
    monitor_thread.join()

    print("\n✓ Ingestion complete.")

In [ ]:
ingest_pdfs(file_paths,qdrant_client,embedder_model,nlp_model)

[MON] sent_q=0  embed_q=0  upsert_q=0 | VRAM 1.11/1.13/16.0 GB (alloc/reserv/total) | RAM 1.08 GB
[MON] sent_q=0  embed_q=0  upsert_q=0 | VRAM 1.11/1.13/16.0 GB (alloc/reserv/total) | RAM 1.08 GB
[MON] sent_q=0  embed_q=0  upsert_q=0 | VRAM 1.11/1.13/16.0 GB (alloc/reserv/total) | RAM 1.08 GB
[MON] sent_q=0  embed_q=0  upsert_q=0 | VRAM 1.11/1.13/16.0 GB (alloc/reserv/total) | RAM 1.08 GB
[MON] sent_q=0  embed_q=0  upsert_q=0 | VRAM 1.11/1.13/16.0 GB (alloc/reserv/total) | RAM 1.08 GB
[MON] sent_q=0  embed_q=0  upsert_q=0 | VRAM 1.11/1.13/16.0 GB (alloc/reserv/total) | RAM 1.08 GB
[MON] sent_q=0  embed_q=0  upsert_q=0 | VRAM 1.11/1.13/16.0 GB (alloc/reserv/total) | RAM 1.08 GB
[MON] sent_q=0  embed_q=0  upsert_q=0 | VRAM 1.11/1.13/16.0 GB (alloc/reserv/total) | RAM 1.08 GB
[MON] sent_q=0  embed_q=0  upsert_q=0 | VRAM 1.11/1.13/16.0 GB (alloc/reserv/total) | RAM 1.08 GB
[MON] sent_q=0  embed_q=0  upsert_q=0 | VRAM 1.11/1.13/16.0 GB (alloc/reserv/total) | RAM 1.08 GB
[MON] sent_q=0  embe